## 6.2 双节点基础 FER/BER 扫描

在上一节中，我们学习了双节点系统架构和安全通信原理。本节先演示单次双节点数据交换的完整流程，再通过 SNR 扫描统计 FER 和 Byte BER，建立双节点链路的性能基线。

本节学习大纲如下：

- 双节点数据交换流程演示（广播→接入→数据交换→断开）
- FER/BER SNR 扫描与曲线绘制

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── node.py               <- SleNode: G/T 双节点模型
├── mac/
│   ├── link_manager.py    <- 链路状态机
│   ├── access.py          <- 接入流程管理
│   └── frame.py           <- AsyncDataFrame: MAC 数据帧
├── phy/
│   ├── mac_interface.py   <- mac_to_iq / iq_to_mac
│   └── channel.py         <- ChannelModel: AWGN 加噪
└── sim/
    └── link_sim.py        <- sim_dual_node_link: 双节点 FER 扫描
                               _channel_impair: 信道损伤
```

---

### 1. 双节点数据交换流程演示

SNR=10 dB，MCS=7，逐步骤展示 G 节点与 T 节点的完整交互（建链过程的关键函数可在 05.04 节中查看）：

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode
from nearlink_sdr.sim.link_sim import _channel_impair

snr_db = 10.0
rng = np.random.default_rng(42)
g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

# ---- 创建 G/T 节点 (MCS=7: QPSK 7/8) ----
g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7, max_retransmit=0))
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7, max_retransmit=0))

# ---- 1. G 广播 ----
print("1. G -> Broadcast")
adv_frame = g.start_advertising()
print(f"   Broadcast frame: {'OK' if adv_frame is not None else 'FAIL'}")

# ---- 2. T 扫描 + 接入 ----
print("2. T -> Scan & Connect")
t.start_scanning()
t.connect(g_addr)
g.accept_connection(t_addr, Role.G_NODE)
print(f"   Scheduler active: {0 in g.scheduler.event_schedulers}")
print(f"   G state={g.state.name}, T state={t.state.name}")

# ---- 3. 数据交换 (通过 AWGN 信道) ----
print("3. Data exchange")
payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
g.send(payload)                                          # G 压入发送队列
tx = g.transmit()                                        # G 生成 IQ
rx_iq = _channel_impair(tx.iq, snr_db, "awgn",           # AWGN 加噪
    6.0, 0.0, "none", g._tx_config.sps, rng)
frame = AsyncDataFrame(segment_type=0, data=payload)     # 构造期望帧
rx = t.receive(rx_iq, len(frame.pack()))                 # T 接收并解调
print(f"   Success={'OK' if rx.success else 'FAIL'}  "
      f"data match={rx.data == payload}")

# ---- 4. 断开 ----
print("4. Disconnect")
g.disconnect()
from nearlink_sdr.node import NodeState
print(f"   G state: {g.state.name}  "
      f"(DISCONNECTED={'OK' if g.state == NodeState.DISCONNECTED else 'FAIL'})")

以上过程封装在函数 sim_dual_node_link() 中，可执行以下代码查看源码：

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "3644,3745p"


---

### 2. FER/BER SNR 扫描

调用 `sim_dual_node_link()` 在 0~14 dB 范围内扫描，每 SNR 发送 50 帧（MCS=7，载荷 10 字节），统计 FER 和 Byte BER：

In [ ]:
from nearlink_sdr.sim.link_sim import sim_dual_node_link

snr_range = np.arange(0, 16, 2)   # SNR 0~14 dB
result = sim_dual_node_link(
    snr_range_db=snr_range,        # 扫描范围
    n_frames=100,                   # 每 SNR 50 帧
    mcs_index=7,                   # MCS=7: QPSK 7/8
    payload_size=10,               # 每帧 10 字节载荷
    seed=42)                       # 随机种子

print(f"{'SNR':>5s}  {'FER':>10s}  {'ByteBER':>10s}")
print("-" * 30)
for s, f, b in zip(snr_range, result["fer"], result["byte_ber"]):
    print(f"{s:5.0f}  {f:10.4f}  {b:10.6f}")

---

### 3. FER/BER 对比曲线

FER 与 Byte BER 在数值上的差异反映了帧内错误的分布特性——一帧包含多个字节，一帧出错时可能只有部分字节出错：

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(snr_range, [max(f, 1e-4) for f in result["fer"]],
            "o-", lw=2, label="FER")
ax.semilogy(snr_range, [max(b, 1e-4) for b in result["byte_ber"]],
            "s--", lw=2, label="Byte BER")
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("Error Rate")
ax.set_title("Dual Node FER/BER (MCS=7, 10B payload)")
ax.legend(); ax.grid(True, which="both", ls="--", alpha=0.5)
ax.set_ylim(bottom=1e-4); plt.show()

---

### 4. 实验分析

由得到的图像应不难得到以下结论：Byte BER 曲线始终低于 FER 曲线。这是因为 FER 只要帧内任意 1 个字节出错就判帧失败，而 Byte BER 统计的是所有传输字节中真正出错的比例——帧出错不代表帧内所有字节都出错。两条曲线之间的间距反映了帧内错误的稀疏程度。在中高 SNR 区两线间距拉开，此时大部分帧能成功解码，少数失败帧中也只有部分字节出错。

---

## 课后实践

请补全下方双节点 FER 与 Byte BER 统计中的 **3 处空缺**（每处一行代码），实现帧错误计数、逐字节比对和 Byte BER 计算。

要求：

1. 补全帧失败时的错误计数
2. 补全逐字节比对（区分"帧出错"与"字节出错"）
3. 补全 Byte BER 的计算公式

完成后运行 ，观察 Byte BER 与 FER 的数值差异。

In [ ]:
%%writefile fer_ber_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode
from nearlink_sdr.sim.link_sim import _channel_impair

snr_db = 8.0
n_frames = 50
rng = np.random.default_rng(42)
g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7, max_retransmit=0))
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7, max_retransmit=0))
g.start_advertising(); t.start_scanning()
t.connect(g_addr); g.accept_connection(t_addr, Role.G_NODE)

frame_errors = 0
byte_errors = 0
total_bytes = 0

for _ in range(n_frames):
    payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    g.send(payload)
    tx = g.transmit()
    if tx.iq is None:
        frame_errors += 1; total_bytes += 10; continue
    rx_iq = _channel_impair(tx.iq, snr_db, "awgn",
        6.0, 0.0, "none", g._tx_config.sps, rng)
    frame = AsyncDataFrame(segment_type=0, data=payload)
    rx = t.receive(rx_iq, len(frame.pack()))
    g.process_feedback(rx.success)
    if not rx.success:
        g._qos.arq.on_ack_received()

    total_bytes += len(payload)
    # ==== 补全统计逻辑 ====
    if not rx.success:
        ______________  # 1: 帧错误计数 +1
    # 2: 逐字节比对 payload 与 rx.data，统计错误字节数
    if rx.data is not None:
        byte_errors += ______________
    else:
        byte_errors += len(payload)

# ==== 3: 计算 Byte BER ====
fer = frame_errors / n_frames
byte_ber = ______________

print(f"SNR={snr_db:.0f} dB | FER={fer:.4f} | ByteBER={byte_ber:.6f}")


执行以下命令进行编译并验证结果：


In [ ]:
!python fer_ber_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/06.02_answer.txt
